# Estatística e Probabilidade — Trabalho Avaliativo Final

Este notebook analisa respostas de uma pesquisa sobre a distribuição do tempo entre obrigações, lazer, descanso e sono. O objetivo é descrever a amostra, estimar probabilidades empíricas e aplicar testes estatísticos para investigar relações entre carga de obrigações, lazer, sono e idade.


## 1. Importações e funções auxiliares


In [ ]:
import os
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
from matplotlib.axes import Axes
import numpy as np
import pandas as pd
from rich import box
from rich.align import Align
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from scipy import stats

In [ ]:
CONSOLE_WIDTH = 80
console = Console(width=CONSOLE_WIDTH, highlight=False)

def show_section(title: str) -> None:
    console.print(Align.center(title))


def show_warning(message: str) -> None:
    console.print(
        Panel(
            message,
            title="Atenção",
            title_align="left",
            width=CONSOLE_WIDTH,
        )
    )


def show_info_panel(title: str, message: str) -> None:
    console.print(
        Panel(
            message,
            title=title,
            title_align="left",
            width=CONSOLE_WIDTH,
        )
    )


def format_number(value: float, digits: int = 2) -> str:
    return f"{value:.{digits}f}"


def show_metric_table(title: str, rows: Sequence[tuple[Any, Any]]) -> None:
    table = Table(
        title=title,
        box=box.SIMPLE,
        show_lines=False,
        expand=True,
        width=CONSOLE_WIDTH,
    )
    table.add_column("Indicador")
    table.add_column("Valor", justify="right")

    for label, value in rows:
        table.add_row(str(label), str(value))

    console.print(table)


def show_data_table(
    title: str,
    columns: Sequence[str],
    rows: Sequence[Sequence[Any]],
) -> None:
    table = Table(
        title=title,
        box=box.SIMPLE,
        show_lines=False,
        expand=True,
        width=CONSOLE_WIDTH,
    )

    for column in columns:
        table.add_column(column)

    for row in rows:
        table.add_row(*[str(value) for value in row])

    console.print(table)


def configure_axis(
    ax: Axes,
    title: str,
    xlabel: str | None = None,
    ylabel: str | None = None,
) -> None:
    ax.set_title(title)

    if xlabel is not None:
        ax.set_xlabel(xlabel)

    if ylabel is not None:
        ax.set_ylabel(ylabel)

    ax.grid(axis="y", alpha=0.25)


def mean_confidence_interval(
    data: pd.Series,
    confidence: float = 0.95,
) -> dict[str, Any]:
    sample = data.dropna()
    n = len(sample)
    mean = sample.mean()
    sample_standard_deviation = sample.std(ddof=1)
    standard_error = sample_standard_deviation / np.sqrt(n)
    alpha = 1 - confidence

    if n <= 30:
        critical_value = stats.t.ppf(1 - alpha / 2, df=n - 1)
        distribution_name = "t de Student"
        degrees_of_freedom = n - 1
    else:
        critical_value = stats.norm.ppf(1 - alpha / 2)
        distribution_name = "Normal padrão"
        degrees_of_freedom = None

    margin_error = critical_value * standard_error
    lower_limit = mean - margin_error
    upper_limit = mean + margin_error

    return {
        "n": n,
        "mean": mean,
        "sample_standard_deviation": sample_standard_deviation,
        "standard_error": standard_error,
        "critical_value": critical_value,
        "margin_error": margin_error,
        "lower_limit": lower_limit,
        "upper_limit": upper_limit,
        "distribution_name": distribution_name,
        "degrees_of_freedom": degrees_of_freedom,
    }


def show_hypothesis_result(
    claim: str,
    null_hypothesis: str,
    alternative_hypothesis: str,
    test_description: str,
    statistics_rows: Sequence[tuple[Any, Any]],
    reject_null: bool,
    reject_interpretation: str,
    fail_interpretation: str,
    type_i_error: str,
    type_ii_error: str,
    alpha_value: float = 0.05,
) -> None:
    console.print(
        Panel(
            "\n".join(
                [
                    f"Afirmação\n{claim}",
                    f"Hipóteses\nH0: {null_hypothesis}\nHa: {alternative_hypothesis}",
                    f"Teste usado\n{test_description}",
                    f"Nível de significância\nα = {alpha_value:.2f}",
                ]
            ),
            title="Desenho do teste",
            title_align="left",
            width=CONSOLE_WIDTH,
        )
    )

    console.print("\n")
    show_metric_table("Resultados", statistics_rows)

    decision = "Rejeitamos H0." if reject_null else "Não rejeitamos H0."
    interpretation = reject_interpretation if reject_null else fail_interpretation

    console.print("\n")
    console.print(
        Panel(
            "Regra de decisão:\n"
            "se p-valor ≤ α, rejeitamos H0.\n"
            "se p-valor > α, não rejeitamos H0.\n\n"
            f"{decision}\n{interpretation}",
            title="Decisão",
            title_align="left",
            width=CONSOLE_WIDTH,
        )
    )

    console.print("\n")
    console.print(
        Panel(
            f"Erro Tipo I\n{type_i_error}\n\n"
            f"Erro Tipo II\n{type_ii_error}",
            title="Riscos de decisão",
            title_align="left",
            width=CONSOLE_WIDTH,
        )
    )


## 2. Carregamento dos dados do Google Forms


In [ ]:
csv_path = Path("data/responses.csv")
responses_spreadsheet_url = os.environ["RESPONSES_SPREADSHEET_URL"]

In [ ]:
if not csv_path.exists():
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df_download = pd.read_csv(responses_spreadsheet_url, sep="\t")
    df_download.to_csv(csv_path, index=False)

In [ ]:
df_raw = pd.read_csv(csv_path)

## 3. Padronização das colunas


In [ ]:
base_columns = [
    "timestamp",
    "birth_date",
    "paid_work_hours",
    "study_hours",
    "housework_hours",
    "caregiving_hours",
    "commute_wait_hours",
    "leisure_rest_hours",
    "screen_leisure_hours",
    "sleep_hours_per_night",
    "desired_extra_leisure_hours",
]

df_raw.columns = base_columns
df = df_raw.copy()

## 4. Formatação dos tipos de dados


In [ ]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    dayfirst=True,
    errors="coerce",
)

df["birth_date"] = pd.to_datetime(
    df["birth_date"],
    dayfirst=True,
    errors="coerce",
)

numeric_columns = [
    "paid_work_hours",
    "study_hours",
    "housework_hours",
    "caregiving_hours",
    "commute_wait_hours",
    "leisure_rest_hours",
    "screen_leisure_hours",
    "sleep_hours_per_night",
    "desired_extra_leisure_hours",
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce",
    )

## 5. Inspeção dos dados

A inspeção abaixo verifica o tamanho dos dados, o período de coleta e possíveis inconsistências.


In [ ]:
show_section("INSPEÇÃO INICIAL DOS DADOS")

collection_start = df["timestamp"].min()
collection_end = df["timestamp"].max()
original_variable_count = len(base_columns)

console.print("\n")
show_metric_table(
    "Resumo da coleta",
    [
        ("Total de respostas", len(df)),
        ("Número de colunas", len(df.columns)),
        ("Data da primeira resposta", collection_start.strftime("%d/%m/%Y %H:%M")),
        ("Data da última resposta", collection_end.strftime("%d/%m/%Y %H:%M")),
        ("Número de variáveis originais", original_variable_count),
    ],
)

In [ ]:
missing_values = df.isna().sum()
missing_values = missing_values[missing_values > 0]

if len(missing_values) == 0:
    show_info_panel(
        "Valores ausentes",
        "Não foram identificados valores ausentes após a conversão dos dados.",
    )
else:
    fig, ax = plt.subplots(figsize=(9, 4))
    missing_values.sort_values().plot(kind="barh", ax=ax)
    configure_axis(
        ax,
        "Valores ausentes por coluna",
        xlabel="Quantidade de valores ausentes",
        ylabel="Coluna",
    )
    plt.tight_layout()
    plt.show()

In [ ]:
weekly_hour_columns = [
    "paid_work_hours",
    "study_hours",
    "housework_hours",
    "caregiving_hours",
    "commute_wait_hours",
    "leisure_rest_hours",
    "screen_leisure_hours",
    "desired_extra_leisure_hours",
]

invalid_sleep = df[
    ~df["sleep_hours_per_night"].between(0, 24)
    & df["sleep_hours_per_night"].notna()
]

invalid_weekly_values = {}

for column in weekly_hour_columns:
    invalid_weekly_values[column] = df[
        ~df[column].between(0, 168)
        & df[column].notna()
    ]

invalid_screen_leisure = df[
    (df["screen_leisure_hours"] > df["leisure_rest_hours"])
    & df["screen_leisure_hours"].notna()
    & df["leisure_rest_hours"].notna()
]

has_invalid_weekly_values = any(
    len(rows) > 0 for rows in invalid_weekly_values.values()
)

if (
    len(invalid_sleep) == 0
    and not has_invalid_weekly_values
    and len(invalid_screen_leisure) == 0
):
    show_info_panel(
        "Checagens informativas",
        "Não foram encontradas inconsistências nas regras verificadas.",
    )
else:
    if len(invalid_sleep) > 0:
        show_warning("Existem valores de sono fora do intervalo de 0 a 24 horas por noite.")
        console.print("\n")
        show_data_table(
            "Casos com sono fora do intervalo",
            ["Índice", "Data de nascimento", "Sono por noite"],
            [
                (
                    index,
                    row["birth_date"].strftime("%d/%m/%Y"),
                    format_number(row["sleep_hours_per_night"]),
                )
                for index, row in invalid_sleep.iterrows()
            ],
        )
        console.print("\n")

    for column, rows in invalid_weekly_values.items():
        if len(rows) > 0:
            show_warning(
                f"Existem valores semanais fora do intervalo de 0 a 168 horas na variável {column}."
            )
            console.print("\n")
            show_data_table(
                f"Casos com valores inválidos em {column}",
                ["Índice", "Valor informado"],
                [
                    (
                        index,
                        format_number(row[column]),
                    )
                    for index, row in rows.iterrows()
                ],
            )
            console.print("\n")

    if len(invalid_screen_leisure) > 0:
        show_warning("Existem respostas em que o lazer com telas é maior que o lazer total.")
        console.print("\n")
        show_data_table(
            "Casos com lazer em telas maior que o lazer total",
            ["Índice", "Lazer e descanso", "Lazer com telas"],
            [
                (
                    index,
                    format_number(row["leisure_rest_hours"]),
                    format_number(row["screen_leisure_hours"]),
                )
                for index, row in invalid_screen_leisure.iterrows()
            ],
        )
        console.print("\n")


## 6. Tratamento de valores impossíveis e inconsistentes

Valores impossíveis pelos limites da pergunta ou inconsistentes entre respostas relacionadas foram considerados inválidos apenas nas análises da variável afetada. Valores de sono maiores que 24 horas por noite não foram usados nas análises de sono. Valores semanais fora do intervalo de 0 a 168 horas não foram usados nas análises da respectiva variável. Casos em que o lazer com telas era maior que o lazer total foram desconsiderados nos cálculos relacionados a telas.

Também foi verificado se a soma entre obrigações semanais, lazer e descanso semanal e sono semanal ultrapassava 168 horas, que é o total de horas em uma semana. Quando isso ocorreu, o caso foi desconsiderado nas análises de distribuição semanal de tempo, pois não há base segura para identificar qual resposta específica causou a inconsistência.

Valores extremos, mas ainda possíveis dentro da unidade da pergunta, foram mantidos na análise principal.


In [ ]:
for column in weekly_hour_columns:
    invalid_values = (
        ~df[column].between(0, 168)
        & df[column].notna()
    )
    df.loc[invalid_values, column] = np.nan

invalid_sleep_values = (
    ~df["sleep_hours_per_night"].between(0, 24)
    & df["sleep_hours_per_night"].notna()
)
df.loc[invalid_sleep_values, "sleep_hours_per_night"] = np.nan

invalid_screen_leisure_values = (
    (df["screen_leisure_hours"] > df["leisure_rest_hours"])
    & df["screen_leisure_hours"].notna()
    & df["leisure_rest_hours"].notna()
)
df.loc[invalid_screen_leisure_values, "screen_leisure_hours"] = np.nan

total_obligation_for_budget = (
    df["paid_work_hours"]
    + df["study_hours"]
    + df["housework_hours"]
    + df["caregiving_hours"]
    + df["commute_wait_hours"]
)

weekly_sleep_for_budget = df["sleep_hours_per_night"] * 7

weekly_time_budget = (
    total_obligation_for_budget
    + df["leisure_rest_hours"]
    + weekly_sleep_for_budget
)

invalid_weekly_time_budget = (
    (weekly_time_budget > 168)
    & weekly_time_budget.notna()
)

if invalid_weekly_time_budget.sum() > 0:
    show_warning(
        "Existem respostas em que obrigações, lazer e sono somam mais de 168 horas semanais."
    )
    console.print("\n")
    show_data_table(
        "Casos com soma semanal maior que 168 horas",
        ["Índice", "Obrigações", "Lazer/descanso", "Sono semanal", "Total"],
        [
            (
                index,
                format_number(total_obligation_for_budget.loc[index]),
                format_number(df.loc[index, "leisure_rest_hours"]),
                format_number(weekly_sleep_for_budget.loc[index]),
                format_number(weekly_time_budget.loc[index]),
            )
            for index in df.index[invalid_weekly_time_budget]
        ],
    )
    console.print("\n")

show_metric_table(
    "Tratamento de valores impossíveis e inconsistentes",
    [
        ("Valores inválidos em sono por noite", len(invalid_sleep)),
        (
            "Valores semanais inválidos",
            sum(len(rows) for rows in invalid_weekly_values.values()),
        ),
        (
            "Casos inválidos em lazer com telas",
            int(invalid_screen_leisure_values.sum()),
        ),
        (
            "Casos com soma semanal maior que 168 horas",
            int(invalid_weekly_time_budget.sum()),
        ),
    ],
)

## 7. Criação das variáveis derivadas


In [ ]:
df["age"] = (
    (df["timestamp"] - df["birth_date"]).dt.days / 365.2425
).round(2)

df["total_obligation_hours"] = total_obligation_for_budget

invalid_total_obligation = (
    (df["total_obligation_hours"] > 168)
    & df["total_obligation_hours"].notna()
)

df.loc[
    invalid_total_obligation | invalid_weekly_time_budget,
    "total_obligation_hours",
] = np.nan

df["weekly_sleep_hours"] = df["sleep_hours_per_night"] * 7

df.loc[
    invalid_weekly_time_budget,
    "weekly_sleep_hours",
] = np.nan

valid_screen_leisure = (
    (df["leisure_rest_hours"] > 0)
    & df["screen_leisure_hours"].notna()
    & df["leisure_rest_hours"].notna()
    & (df["screen_leisure_hours"] <= df["leisure_rest_hours"])
    & ~invalid_weekly_time_budget
)

df["screen_leisure_percentage"] = np.where(
    valid_screen_leisure,
    df["screen_leisure_hours"] / df["leisure_rest_hours"] * 100,
    np.nan,
)

df["leisure_obligation_ratio"] = np.where(
    (df["total_obligation_hours"] > 0)
    & df["leisure_rest_hours"].notna()
    & ~invalid_weekly_time_budget,
    df["leisure_rest_hours"] / df["total_obligation_hours"],
    np.nan,
)

derived_variables = [
    "age",
    "total_obligation_hours",
    "weekly_sleep_hours",
    "screen_leisure_percentage",
    "leisure_obligation_ratio",
]

In [ ]:
show_data_table(
    "Variáveis derivadas",
    ["Variável", "Significado"],
    [
        ("age", "Idade no momento da resposta."),
        ("total_obligation_hours", "Soma de trabalho, estudo, casa, cuidado e deslocamento, excluindo casos com orçamento semanal inconsistente."),
        ("weekly_sleep_hours", "Sono médio por noite multiplicado por 7, excluindo casos com orçamento semanal inconsistente."),
        ("screen_leisure_percentage", "Percentual do lazer e descanso feito com telas, calculado apenas para casos consistentes."),
        ("leisure_obligation_ratio", "Proporção entre lazer/descanso e obrigações, calculada apenas para casos consistentes."),
    ],
)

console.print("\n")
show_metric_table(
    "Resumo dos dados tratados",
    [
        ("Total de respostas", len(df)),
        ("Data da primeira resposta", collection_start.strftime("%d/%m/%Y %H:%M")),
        ("Data da última resposta", collection_end.strftime("%d/%m/%Y %H:%M")),
        ("Número de variáveis originais", original_variable_count),
        ("Número de variáveis derivadas", len(derived_variables)),
    ],
)

## 8. Análise exploratória e descritiva

Esta parte resume as principais variáveis quantitativas da pesquisa. A tabela apresenta medidas de posição e dispersão, enquanto os gráficos ajudam a visualizar a distribuição das horas dedicadas a obrigações, lazer e sono.


In [ ]:
main_variables = [
    "age",
    "paid_work_hours",
    "study_hours",
    "housework_hours",
    "caregiving_hours",
    "commute_wait_hours",
    "total_obligation_hours",
    "leisure_rest_hours",
    "screen_leisure_hours",
    "sleep_hours_per_night",
    "desired_extra_leisure_hours",
    "screen_leisure_percentage",
    "leisure_obligation_ratio",
]

variable_labels = {
    "age": "Idade",
    "paid_work_hours": "Trabalho",
    "study_hours": "Estudo",
    "housework_hours": "Casa",
    "caregiving_hours": "Cuidado",
    "commute_wait_hours": "Deslocamento",
    "total_obligation_hours": "Obrigações",
    "leisure_rest_hours": "Lazer/descanso",
    "screen_leisure_hours": "Lazer com telas",
    "sleep_hours_per_night": "Sono por noite",
    "desired_extra_leisure_hours": "Lazer desejado",
    "screen_leisure_percentage": "% lazer com telas",
    "leisure_obligation_ratio": "Lazer/obrigações",
}

summary = pd.DataFrame(
    {
        "count": df[main_variables].count(),
        "mean": df[main_variables].mean(),
        "median": df[main_variables].median(),
        "standard_deviation": df[main_variables].std(),
        "minimum": df[main_variables].min(),
        "q1": df[main_variables].quantile(0.25),
        "q3": df[main_variables].quantile(0.75),
        "maximum": df[main_variables].max(),
    }
)

summary["range"] = summary["maximum"] - summary["minimum"]
summary["interquartile_range"] = summary["q3"] - summary["q1"]
summary = summary.round(2)

summary_display = summary.rename(index=variable_labels).rename(
    columns={
        "count": "Quantidade",
        "mean": "Média",
        "median": "Mediana",
        "standard_deviation": "Desvio padrão",
        "minimum": "Mínimo",
        "q1": "Q1",
        "q3": "Q3",
        "maximum": "Máximo",
        "range": "Amplitude",
        "interquartile_range": "Intervalo interquartil",
    }
)

summary_display

In [ ]:
activity_means = pd.Series(
    {
        "Trabalho": df["paid_work_hours"].mean(),
        "Estudo": df["study_hours"].mean(),
        "Casa": df["housework_hours"].mean(),
        "Cuidado": df["caregiving_hours"].mean(),
        "Deslocamento": df["commute_wait_hours"].mean(),
        "Lazer/descanso": df["leisure_rest_hours"].mean(),
    }
)

fig, ax = plt.subplots(figsize=(9, 5))
activity_means.plot(kind="bar", ax=ax)
configure_axis(
    ax,
    "Média semanal de horas por atividade",
    xlabel="Atividade",
    ylabel="Horas semanais",
)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
boxplot_data = [
    df["total_obligation_hours"].dropna(),
    df["leisure_rest_hours"].loc[~invalid_weekly_time_budget].dropna(),
    df["weekly_sleep_hours"].dropna(),
]

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(
    boxplot_data,
    tick_labels=["Obrigações", "Lazer/descanso", "Sono semanal"],
)
configure_axis(
    ax,
    "Comparação entre obrigações, lazer e sono",
    xlabel="Variável",
    ylabel="Horas semanais",
)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(
    df["total_obligation_hours"].dropna(),
    bins=8,
    edgecolor="black",
)
configure_axis(
    ax,
    "Distribuição do total semanal de obrigações",
    xlabel="Horas semanais de obrigações",
    ylabel="Número de participantes",
)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(
    df["leisure_rest_hours"].loc[~invalid_weekly_time_budget].dropna(),
    bins=8,
    edgecolor="black",
)
configure_axis(
    ax,
    "Distribuição do tempo semanal de lazer e descanso",
    xlabel="Horas semanais de lazer e descanso",
    ylabel="Número de participantes",
)
plt.tight_layout()
plt.show()

## 9. Probabilidades empíricas

As probabilidades empíricas foram calculadas por frequência relativa, isto é, pela razão entre o número de participantes que satisfazem determinado evento e o total de participantes considerados.

$$P(A) \approx \frac{n(A)}{n}$$

Para as probabilidades condicionais:

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)},\quad P(B) > 0$$

No caso da amostra, a probabilidade condicional foi calculada como a proporção de participantes que desejam mais lazer dentro de cada grupo de carga de obrigações.


In [ ]:
sleep_less_than_7_probability = (df["sleep_hours_per_night"] < 7).mean()

wants_more_leisure_probability = (
    df["desired_extra_leisure_hours"] > 0
).mean()

mostly_screen_leisure_probability = (
    df.loc[valid_screen_leisure, "screen_leisure_percentage"] > 50
).mean()

median_obligation_hours = df["total_obligation_hours"].median()

high_obligation_group = df[
    df["total_obligation_hours"] > median_obligation_hours
]

low_obligation_group = df[
    df["total_obligation_hours"] <= median_obligation_hours
]

wants_more_leisure_high_obligation_probability = (
    high_obligation_group["desired_extra_leisure_hours"] > 0
).mean()

wants_more_leisure_low_obligation_probability = (
    low_obligation_group["desired_extra_leisure_hours"] > 0
).mean()

probability_labels = [
    "Dormir menos de 7 horas por noite",
    "Desejar mais lazer ou descanso",
    "Mais da metade do lazer ser com telas",
    "Desejar mais lazer | alta carga de obrigações",
    "Desejar mais lazer | baixa carga de obrigações",
]

probability_values = [
    sleep_less_than_7_probability,
    wants_more_leisure_probability,
    mostly_screen_leisure_probability,
    wants_more_leisure_high_obligation_probability,
    wants_more_leisure_low_obligation_probability,
]

show_metric_table(
    "Probabilidades empíricas",
    [
        (label, f"{value:.2%}")
        for label, value in zip(probability_labels, probability_values)
    ],
)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
probability_percentages = pd.Series(
    [value * 100 for value in probability_values],
    index=[
        "Sono < 7h",
        "Deseja mais lazer",
        "Telas > 50%",
        "Mais lazer | alta obrigação",
        "Mais lazer | baixa obrigação",
    ],
)
probability_percentages.plot(kind="bar", ax=ax)
configure_axis(
    ax,
    "Probabilidades empíricas observadas",
    xlabel="Evento",
    ylabel="Percentual da amostra",
)
ax.set_ylim(0, 100)
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

## 10. Intervalos de confiança

O intervalo de confiança segue:

$$\bar{x} - e \leq \mu \leq \bar{x} + e$$

ou

$$\mu = \bar{x} \pm e$$

Nessa expressão, $\bar{x}$ é a média amostral, $\mu$ é a média populacional, $e$ é a margem de erro e $1 - \alpha$ é o nível de confiança. Como o desvio padrão populacional não é conhecido, utiliza-se o desvio padrão amostral.

A distribuição normal é usada como referência para médias amostrais em amostras grandes. Quando a amostra é pequena e o desvio padrão populacional é desconhecido, utiliza-se a distribuição t de Student.


In [ ]:
leisure_interval = mean_confidence_interval(
    df["leisure_rest_hours"].loc[~invalid_weekly_time_budget]
)
sleep_interval = mean_confidence_interval(df["sleep_hours_per_night"])


def format_interval_value(interval: Mapping[str, Any], key: str) -> str:
    return format_number(interval[key])


def format_degrees_of_freedom(interval: Mapping[str, Any]) -> str:
    if interval["degrees_of_freedom"] is None:
        return "-"

    return str(interval["degrees_of_freedom"])


show_data_table(
    "Intervalos de confiança (95%)",
    ["Indicador", "Lazer e descanso semanal", "Sono por noite"],
    [
        ("n", leisure_interval["n"], sleep_interval["n"]),
        ("Média", format_interval_value(leisure_interval, "mean"), format_interval_value(sleep_interval, "mean")),
        (
            "Desvio padrão amostral",
            format_interval_value(leisure_interval, "sample_standard_deviation"),
            format_interval_value(sleep_interval, "sample_standard_deviation"),
        ),
        (
            "Erro padrão",
            format_interval_value(leisure_interval, "standard_error"),
            format_interval_value(sleep_interval, "standard_error"),
        ),
        ("Distribuição usada", leisure_interval["distribution_name"], sleep_interval["distribution_name"]),
        ("Graus de liberdade", format_degrees_of_freedom(leisure_interval), format_degrees_of_freedom(sleep_interval)),
        (
            "Valor crítico",
            format_interval_value(leisure_interval, "critical_value"),
            format_interval_value(sleep_interval, "critical_value"),
        ),
        (
            "Margem de erro",
            format_interval_value(leisure_interval, "margin_error"),
            format_interval_value(sleep_interval, "margin_error"),
        ),
        (
            "Intervalo de confiança",
            f"({leisure_interval['lower_limit']:.2f}, {leisure_interval['upper_limit']:.2f})",
            f"({sleep_interval['lower_limit']:.2f}, {sleep_interval['upper_limit']:.2f})",
        ),
    ],
)

console.print("\n")
show_info_panel(
    "Interpretação",
    "O intervalo apresenta uma faixa plausível para a média populacional, considerando a amostra coletada, a margem de erro e o nível de confiança adotado.",
)

## 11. Testes de hipótese

Os testes de hipótese foram usados para avaliar afirmações sobre os dados coletados. Em cada teste, a hipótese nula representa a situação inicial considerada verdadeira, enquanto a hipótese alternativa representa a afirmação que se deseja investigar.

O nível de significância adotado foi:

$$\alpha = 0{,}05$$

A decisão foi tomada pela comparação entre o p-valor e $\alpha$:

$$p\text{-valor} \leq \alpha \Rightarrow \text{rejeitamos } H_0$$

$$p\text{-valor} > \alpha \Rightarrow \text{não rejeitamos } H_0$$

Nos testes de correlação, o coeficiente analisado é $r$, que representa a correlação de Pearson. Valores positivos indicam associação linear positiva, valores negativos indicam associação linear negativa e valores próximos de zero indicam pouca associação linear.


In [ ]:
alpha = 0.05
test_summary_rows = []

### 11.1 Obrigações e desejo de mais lazer

A afirmação investigada é que participantes com mais horas semanais de obrigações tendem a desejar mais horas adicionais de lazer ou descanso.

Como a afirmação indica uma relação positiva, o teste é unilateral à direita. O coeficiente analisado é $r$, que representa a correlação de Pearson entre o total semanal de obrigações e o desejo de horas adicionais de lazer.

As hipóteses são:

$$H_0: r \leq 0$$

$$H_a: r > 0$$

Nesse caso, rejeitar $H_0$ significa obter evidência estatística suficiente para apoiar a afirmação de que maiores cargas de obrigações estão associadas a maior desejo de lazer ou descanso.


In [ ]:
show_section("TESTE 1 - OBRIGAÇÕES E DESEJO DE MAIS LAZER")

test_1_data = df[
    [
        "total_obligation_hours",
        "desired_extra_leisure_hours",
    ]
].dropna()

test_1_r, test_1_p_two_sided = stats.pearsonr(
    test_1_data["total_obligation_hours"],
    test_1_data["desired_extra_leisure_hours"],
)

if test_1_r > 0:
    test_1_p_one_sided = test_1_p_two_sided / 2
else:
    test_1_p_one_sided = 1 - test_1_p_two_sided / 2

test_1_reject_null = test_1_p_one_sided <= alpha

console.print("\n")
show_hypothesis_result(
    claim=(
        "Participantes com mais horas semanais de obrigações tendem a desejar "
        "mais horas adicionais de lazer ou descanso."
    ),
    null_hypothesis="r ≤ 0",
    alternative_hypothesis="r > 0",
    test_description="Correlação de Pearson com teste unilateral à direita.",
    statistics_rows=[
        ("Tamanho da amostra", len(test_1_data)),
        ("Correlação de Pearson r", round(test_1_r, 3)),
        ("p-valor unilateral", round(test_1_p_one_sided, 5)),
        (
            "Comparação",
            f"p = {test_1_p_one_sided:.5f} "
            f"{'≤' if test_1_p_one_sided <= alpha else '>'} "
            f"α = {alpha:.2f}",
        ),
    ],
    reject_null=test_1_reject_null,
    reject_interpretation=(
        "Há evidência estatística suficiente para apoiar a afirmação de que "
        "participantes com mais obrigações tendem a desejar mais horas adicionais "
        "de lazer ou descanso."
    ),
    fail_interpretation=(
        "Não há evidência estatística suficiente para apoiar a afirmação de que "
        "participantes com mais obrigações tendem a desejar mais horas adicionais "
        "de lazer ou descanso."
    ),
    type_i_error=(
        "Rejeitar H0 sendo que ela era verdadeira. "
        "Nesse caso, seria concluir que participantes com mais obrigações tendem "
        "a desejar mais lazer quando essa associação positiva não existe."
    ),
    type_ii_error=(
        "Não rejeitar H0 sendo que ela era falsa. "
        "Nesse caso, seria deixar de identificar uma associação positiva real "
        "entre obrigações e desejo de mais lazer."
    ),
)

test_summary_rows.append(
    (
        "Obrigações e desejo de mais lazer",
        f"r = {test_1_r:.3f}",
        f"{test_1_p_one_sided:.5f}",
        "Rejeitamos H0" if test_1_reject_null else "Não rejeitamos H0",
    )
)

### 11.2 Obrigações e tempo de lazer

A afirmação investigada é que existe relação entre o total semanal de obrigações e o tempo semanal de lazer e descanso.

Como a afirmação não define previamente se a relação seria positiva ou negativa, o teste é bilateral. O coeficiente analisado é $r$, que representa a correlação de Pearson entre essas duas variáveis.

As hipóteses são:

$$H_0: r = 0$$

$$H_a: r \neq 0$$

Nesse caso, rejeitar $H_0$ significa obter evidência estatística suficiente para afirmar que há relação linear entre obrigações e tempo de lazer na amostra.


In [ ]:
show_section("TESTE 2 - OBRIGAÇÕES E TEMPO DE LAZER")

test_2_data = df[
    [
        "total_obligation_hours",
        "leisure_rest_hours",
    ]
].dropna()

test_2_r, test_2_p_two_sided = stats.pearsonr(
    test_2_data["total_obligation_hours"],
    test_2_data["leisure_rest_hours"],
)

test_2_reject_null = test_2_p_two_sided <= alpha

console.print("\n")
show_hypothesis_result(
    claim=(
        "Existe relação entre o total semanal de obrigações e o tempo semanal "
        "de lazer e descanso."
    ),
    null_hypothesis="r = 0",
    alternative_hypothesis="r ≠ 0",
    test_description="Correlação de Pearson com teste bilateral.",
    statistics_rows=[
        ("Tamanho da amostra", len(test_2_data)),
        ("Correlação de Pearson r", round(test_2_r, 3)),
        ("p-valor", round(test_2_p_two_sided, 5)),
        (
            "Comparação",
            f"p = {test_2_p_two_sided:.5f} "
            f"{'≤' if test_2_p_two_sided <= alpha else '>'} "
            f"α = {alpha:.2f}",
        ),
    ],
    reject_null=test_2_reject_null,
    reject_interpretation=(
        "Há evidência estatística suficiente para afirmar que existe relação "
        "linear entre obrigações e tempo de lazer na amostra."
    ),
    fail_interpretation=(
        "Não há evidência estatística suficiente para afirmar que existe relação "
        "linear entre obrigações e tempo de lazer na amostra."
    ),
    type_i_error=(
        "Rejeitar H0 sendo que ela era verdadeira. "
        "Nesse caso, seria concluir que existe relação linear entre obrigações "
        "e tempo de lazer quando essa relação não existe."
    ),
    type_ii_error=(
        "Não rejeitar H0 sendo que ela era falsa. "
        "Nesse caso, seria deixar de identificar uma relação linear real entre "
        "obrigações e tempo de lazer."
    ),
)

test_summary_rows.append(
    (
        "Obrigações e tempo de lazer",
        f"r = {test_2_r:.3f}",
        f"{test_2_p_two_sided:.5f}",
        "Rejeitamos H0" if test_2_reject_null else "Não rejeitamos H0",
    )
)

### 11.3 Média de sono menor que 7 horas

A afirmação investigada é que a média de sono por noite dos participantes é inferior a 7 horas. Esse valor foi adotado como referência porque representa um limite mínimo usado em recomendações gerais de sono.

Como o objetivo é verificar se a média está abaixo desse valor de referência, o teste é unilateral à esquerda.

Como o desvio padrão populacional não é conhecido, utiliza-se o desvio padrão amostral. Nesse caso, a distribuição t é usada com \(n - 1\) graus de liberdade.

O desvio padrão amostral é calculado por:

$$s = \sqrt{\frac{\sum (x_i - \bar{x})^2}{n - 1}}$$

A estatística de teste é:

$$t = \frac{\bar{x} - \mu_0}{\frac{s}{\sqrt{n}}}$$

Na fórmula, $\bar{x}$ representa a média amostral de sono, $\mu_0 = 7$ é o valor de referência, $s$ é o desvio padrão amostral e $n$ é o tamanho da amostra.

As hipóteses são:

$$H_0: \mu \geq 7$$

$$H_a: \mu < 7$$

Nesse caso, rejeitar $H_0$ significa obter evidência estatística suficiente para apoiar a afirmação de que a média de sono da amostra é menor que 7 horas por noite.


In [ ]:
show_section("TESTE 3 - MÉDIA DE SONO MENOR QUE 7 HORAS")

test_3_data = df["sleep_hours_per_night"].dropna()

test_3_t, test_3_p_two_sided = stats.ttest_1samp(
    test_3_data,
    popmean=7,
)

if test_3_t < 0:
    test_3_p_one_sided = test_3_p_two_sided / 2
else:
    test_3_p_one_sided = 1 - test_3_p_two_sided / 2

test_3_reject_null = test_3_p_one_sided <= alpha

console.print("\n")
show_hypothesis_result(
    claim="A média de sono dos participantes é menor que 7 horas por noite.",
    null_hypothesis="μ ≥ 7",
    alternative_hypothesis="μ < 7",
    test_description="Teste t unilateral à esquerda para uma média.",
    statistics_rows=[
        ("Tamanho da amostra", len(test_3_data)),
        ("Média observada", round(test_3_data.mean(), 2)),
        ("Desvio padrão amostral", round(test_3_data.std(ddof=1), 2)),
        ("Graus de liberdade", len(test_3_data) - 1),
        ("Estatística t", round(test_3_t, 3)),
        ("p-valor unilateral", round(test_3_p_one_sided, 5)),
        (
            "Comparação",
            f"p = {test_3_p_one_sided:.5f} "
            f"{'≤' if test_3_p_one_sided <= alpha else '>'} "
            f"α = {alpha:.2f}",
        ),
    ],
    reject_null=test_3_reject_null,
    reject_interpretation=(
        "Há evidência estatística suficiente para apoiar a afirmação de que "
        "a média de sono dos participantes é menor que 7 horas por noite."
    ),
    fail_interpretation=(
        "Não há evidência estatística suficiente para apoiar a afirmação de que "
        "a média de sono dos participantes é menor que 7 horas por noite."
    ),
    type_i_error=(
        "Rejeitar H0 sendo que ela era verdadeira. "
        "Nesse caso, seria concluir que a média de sono é menor que 7 horas "
        "quando ela é de pelo menos 7 horas."
    ),
    type_ii_error=(
        "Não rejeitar H0 sendo que ela era falsa. "
        "Nesse caso, seria deixar de identificar que a média de sono é realmente "
        "menor que 7 horas."
    ),
)

test_summary_rows.append(
    (
        "Média de sono menor que 7 horas",
        f"t = {test_3_t:.3f}",
        f"{test_3_p_one_sided:.5f}",
        "Rejeitamos H0" if test_3_reject_null else "Não rejeitamos H0",
    )
)

### 11.4 Idade e total de obrigações

A afirmação investigada é que existe relação entre a idade dos participantes e o total semanal de obrigações.

Como a afirmação não define previamente se a relação seria positiva ou negativa, o teste é bilateral. O coeficiente analisado é $r$, que representa a correlação de Pearson entre idade e total semanal de obrigações.

As hipóteses são:

$$H_0: r = 0$$

$$H_a: r \neq 0$$

Nesse caso, rejeitar $H_0$ significa obter evidência estatística suficiente para afirmar que existe relação linear entre idade e total de obrigações na amostra.


In [ ]:
show_section("TESTE 4 - IDADE E TOTAL DE OBRIGAÇÕES")

test_4_data = df[
    [
        "age",
        "total_obligation_hours",
    ]
].dropna()

test_4_r, test_4_p_two_sided = stats.pearsonr(
    test_4_data["age"],
    test_4_data["total_obligation_hours"],
)

test_4_reject_null = test_4_p_two_sided <= alpha

console.print("\n")
show_hypothesis_result(
    claim=(
        "Existe relação entre a idade dos participantes e o total semanal "
        "de obrigações."
    ),
    null_hypothesis="r = 0",
    alternative_hypothesis="r ≠ 0",
    test_description="Correlação de Pearson com teste bilateral.",
    statistics_rows=[
        ("Tamanho da amostra", len(test_4_data)),
        ("Correlação de Pearson r", round(test_4_r, 3)),
        ("p-valor", round(test_4_p_two_sided, 5)),
        (
            "Comparação",
            f"p = {test_4_p_two_sided:.5f} "
            f"{'≤' if test_4_p_two_sided <= alpha else '>'} "
            f"α = {alpha:.2f}",
        ),
    ],
    reject_null=test_4_reject_null,
    reject_interpretation=(
        "Há evidência estatística suficiente para afirmar que existe relação "
        "linear entre idade e total de obrigações na amostra."
    ),
    fail_interpretation=(
        "Não há evidência estatística suficiente para afirmar que existe relação "
        "linear entre idade e total de obrigações na amostra."
    ),
    type_i_error=(
        "Rejeitar H0 sendo que ela era verdadeira. "
        "Nesse caso, seria concluir que existe relação linear entre idade e total "
        "de obrigações quando essa relação não existe."
    ),
    type_ii_error=(
        "Não rejeitar H0 sendo que ela era falsa. "
        "Nesse caso, seria deixar de identificar uma relação linear real entre "
        "idade e total de obrigações."
    ),
)

test_summary_rows.append(
    (
        "Idade e total de obrigações",
        f"r = {test_4_r:.3f}",
        f"{test_4_p_two_sided:.5f}",
        "Rejeitamos H0" if test_4_reject_null else "Não rejeitamos H0",
    )
)

### Resumo dos testes

A tabela abaixo reúne a estatística calculada, o p-valor e a decisão tomada em cada teste. Esse resumo permite comparar rapidamente quais afirmações tiveram evidência estatística suficiente na amostra.


In [ ]:
show_data_table(
    "Resumo dos testes de hipótese",
    ["Teste", "Estatística", "p-valor", "Decisão"],
    test_summary_rows,
)

## 12. Regressão linear simples

A regressão estima a associação entre carga total de obrigações e horas adicionais desejadas de lazer ou descanso. Como a pesquisa é observacional, o resultado indica associação na amostra, não causalidade.

O modelo segue a forma:

$$Y' = mX + b$$

Nesse modelo, $X$ é o total semanal de obrigações, $Y'$ é o valor estimado de horas adicionais desejadas de lazer, $m$ é a inclinação da reta e $b$ é o intercepto.

O coeficiente de determinação indica a proporção da variação de $Y$ que é explicada pelo modelo linear. Ele pode ser calculado como o quadrado do coeficiente de correlação.

$$r^2 = \frac{\text{variação explicada}}{\text{variação total}}$$

$$r^2 = r \times r$$

O erro padrão da estimativa mede o tamanho típico dos desvios entre os valores observados e os valores estimados pela reta.

$$s_e = \sqrt{\frac{\sum (y_i - \hat{y}_i)^2}{n - 2}}$$


In [ ]:
show_section("REGRESSÃO LINEAR - OBRIGAÇÕES E DESEJO DE MAIS LAZER")

regression_x = test_1_data["total_obligation_hours"]
regression_y = test_1_data["desired_extra_leisure_hours"]

regression_1 = stats.linregress(
    regression_x,
    regression_y,
)

predicted_y = regression_1.intercept + regression_1.slope * regression_x
residuals = regression_y - predicted_y

total_variation = ((regression_y - regression_y.mean()) ** 2).sum()
unexplained_variation = (residuals ** 2).sum()
explained_variation = ((predicted_y - regression_y.mean()) ** 2).sum()

r_squared_from_variation = explained_variation / total_variation
r_squared_from_correlation = regression_1.rvalue ** 2

regression_sample_size = len(test_1_data)
standard_error_estimate = np.sqrt(
    unexplained_variation / (regression_sample_size - 2)
)

regression_equation = (
    "Horas adicionais de lazer = "
    f"{regression_1.intercept:.2f} + "
    f"({regression_1.slope:.2f} × total de obrigações)"
)

if regression_1.slope >= 0:
    slope_interpretation = (
        f"A cada hora adicional de obrigações, o modelo estima aumento médio "
        f"de {regression_1.slope:.2f} hora no desejo de lazer adicional."
    )
else:
    slope_interpretation = (
        f"A cada hora adicional de obrigações, o modelo estima redução média "
        f"de {abs(regression_1.slope):.2f} hora no desejo de lazer adicional."
    )

r_squared_interpretation = (
    f"Aproximadamente {r_squared_from_correlation * 100:.2f}% da variação "
    f"no desejo de lazer adicional é explicada pelo modelo linear com o total de obrigações."
)

console.print("\n")
show_metric_table(
    "Regressão linear simples",
    [
        ("Equação estimada", regression_equation),
        ("Inclinação m", round(regression_1.slope, 3)),
        ("Intercepto b", round(regression_1.intercept, 3)),
        ("Interpretação da inclinação", slope_interpretation),
        ("Variação total", round(total_variation, 3)),
        ("Variação explicada", round(explained_variation, 3)),
        ("Variação não explicada", round(unexplained_variation, 3)),
        ("R² pela correlação", round(r_squared_from_correlation, 3)),
        ("R² pela razão das variações", round(r_squared_from_variation, 3)),
        ("Interpretação de R²", r_squared_interpretation),
        ("Erro padrão da estimativa", round(standard_error_estimate, 3)),
        ("p-valor", round(regression_1.pvalue, 5)),
    ],
)

In [ ]:
x_order = np.argsort(regression_x.to_numpy())
x_sorted = regression_x.to_numpy()[x_order]
y_estimated_sorted = predicted_y.to_numpy()[x_order]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(regression_x, regression_y, label="Valores observados")
ax.plot(x_sorted, y_estimated_sorted, label="Reta estimada")
configure_axis(
    ax,
    "Obrigações e desejo de mais lazer",
    xlabel="Total semanal de obrigações",
    ylabel="Horas adicionais de lazer desejadas",
)
ax.legend()
plt.tight_layout()
plt.show()

## 13. Visualizações complementares

Os gráficos abaixo mostram relações entre as variáveis centrais da pesquisa. Eles ajudam a observar padrões visuais, mas não substituem os testes de hipótese.


### Obrigações e tempo de lazer

Este gráfico compara a carga total de obrigações com o tempo semanal de lazer e descanso informado pelos participantes.


In [ ]:
regression_2 = stats.linregress(
    test_2_data["total_obligation_hours"],
    test_2_data["leisure_rest_hours"],
)

x = test_2_data["total_obligation_hours"]
y = test_2_data["leisure_rest_hours"]

x_order = np.argsort(x.to_numpy())
x_sorted = x.to_numpy()[x_order]
y_estimated_sorted = regression_2.intercept + regression_2.slope * x_sorted

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y)
ax.plot(x_sorted, y_estimated_sorted)
configure_axis(
    ax,
    "Obrigações e tempo de lazer",
    xlabel="Total semanal de obrigações",
    ylabel="Horas semanais de lazer e descanso",
)
plt.tight_layout()
plt.show()

### Idade e obrigações

A dispersão mostra se participantes mais velhos tendem a concentrar maior carga semanal de obrigações.


In [ ]:
regression_3 = stats.linregress(
    test_4_data["age"],
    test_4_data["total_obligation_hours"],
)

x = test_4_data["age"]
y = test_4_data["total_obligation_hours"]

x_order = np.argsort(x.to_numpy())
x_sorted = x.to_numpy()[x_order]
y_estimated_sorted = regression_3.intercept + regression_3.slope * x_sorted

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y)
ax.plot(x_sorted, y_estimated_sorted)
configure_axis(
    ax,
    "Idade e total de obrigações",
    xlabel="Idade em anos",
    ylabel="Total semanal de obrigações",
)
plt.tight_layout()
plt.show()

### Matriz de correlação

A matriz resume as correlações lineares entre variáveis quantitativas. Valores próximos de 1 indicam associação linear positiva, próximos de -1 indicam associação linear negativa e próximos de 0 indicam pouca associação linear.


In [ ]:
correlation_columns = [
    "age",
    "paid_work_hours",
    "study_hours",
    "housework_hours",
    "caregiving_hours",
    "commute_wait_hours",
    "total_obligation_hours",
    "leisure_rest_hours",
    "sleep_hours_per_night",
    "desired_extra_leisure_hours",
]

correlation_matrix = df[correlation_columns].corr()

display_labels = [
    "Idade",
    "Trabalho",
    "Estudo",
    "Casa",
    "Cuidado",
    "Deslocamento",
    "Obrigações",
    "Lazer",
    "Sono",
    "Lazer desejado",
]

fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(correlation_matrix)

ax.set_xticks(range(len(display_labels)))
ax.set_xticklabels(display_labels, rotation=90)
ax.set_yticks(range(len(display_labels)))
ax.set_yticklabels(display_labels)

for row in range(len(display_labels)):
    for column in range(len(display_labels)):
        ax.text(
            column,
            row,
            f"{correlation_matrix.iloc[row, column]:.2f}",
            ha="center",
            va="center",
            fontsize=8,
        )

fig.colorbar(image, ax=ax)
ax.set_title("Matriz de correlação")
plt.tight_layout()
plt.show()

## 14. Conclusão principal

A conclusão abaixo é derivada principalmente do teste entre carga de obrigações e desejo de mais lazer ou descanso.


In [ ]:
if test_1_reject_null:
    main_conclusion = (
        "Com base no teste principal, rejeitamos H0. Há evidência estatística "
        "de que, na amostra, participantes com maior carga de obrigações tendem "
        "a desejar mais horas adicionais de lazer ou descanso."
    )
else:
    main_conclusion = (
        "Com base no teste principal, não rejeitamos H0. Não há evidência "
        "estatística suficiente para afirmar que, na amostra, participantes com maior "
        "carga de obrigações tendem a desejar mais horas adicionais de lazer ou descanso."
    )

secondary_conclusions = []

if test_2_reject_null:
    secondary_conclusions.append(
        "Também houve evidência de relação linear entre obrigações e tempo de lazer."
    )
else:
    secondary_conclusions.append(
        "Não houve evidência estatística suficiente de relação linear entre obrigações e tempo de lazer."
    )

if test_3_reject_null:
    secondary_conclusions.append(
        "Também houve evidência de que a média de sono é menor que 7 horas por noite."
    )
else:
    secondary_conclusions.append(
        "Não houve evidência estatística suficiente de que a média de sono seja menor que 7 horas por noite."
    )

if test_4_reject_null:
    secondary_conclusions.append(
        "Além disso, houve evidência de relação linear entre idade e total de obrigações."
    )
else:
    secondary_conclusions.append(
        "Não houve evidência estatística suficiente de relação linear entre idade e total de obrigações."
    )

show_info_panel(
    "Conclusão principal",
    main_conclusion + "\n\n" + " ".join(secondary_conclusions),
)